# 🎨 TRELLIS (Image-to-3D) — Google Colab Setup

**Official repo:** https://github.com/microsoft/TRELLIS

This notebook installs TRELLIS the way the **official `setup.sh` script is designed to be used**:
it creates its own dedicated `trellis` conda environment pinned to **Python 3.10**, and installs
the exact pinned `PyTorch 2.4.0 + CUDA 11.8` inside that environment.

**Why this matters on Colab specifically:** Colab's base Python is now **3.13**. PyTorch 2.4.0 and
several of TRELLIS's dependencies (`open3d`, `kaolin`, etc.) don't have wheels for 3.13. Installing
into Colab's base environment (skipping `--new-env`) leads to a cascade of version-mismatch errors.
Using `--new-env` sidesteps all of that, because the official script always targets Python 3.10
regardless of what Colab's base interpreter is.

### How to run this notebook
1. `Runtime → Change runtime type → GPU` (T4/L4/A100 — at least 16GB VRAM recommended).
2. Run cells **top to bottom, in order**.
3. **One cell restarts the kernel automatically** (the `condacolab` install). This is expected —
   just continue running from the next cell down afterward. Do not re-run the condacolab cell.
4. The full setup (CUDA toolkit + conda env + all TRELLIS extensions) takes roughly **20–40 minutes**.


## 1. Check GPU
Make sure a GPU is attached before doing anything else.

In [2]:
!nvidia-smi

Tue Sep  1 05:16:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install CUDA 11.8 Toolkit (nvcc)

Colab's GPU runtime ships NVIDIA *drivers* but not necessarily a matching CUDA *Toolkit* with `nvcc`.
Several TRELLIS extensions (`diffoctreerast`, `kaolin`, `nvdiffrast`) compile CUDA code at install
time and need `nvcc` from a toolkit that matches the CUDA 11.8 PyTorch build TRELLIS pins to.

This installs CUDA 11.8 **alongside** whatever Colab already has — it does not remove the driver.

In [3]:
%%bash
set -e

echo '========================================'
echo '🔧 Installing CUDA 11.8 Toolkit (nvcc)'
echo '========================================'

if [ -x /usr/local/cuda-11.8/bin/nvcc ]; then
    echo "✅ CUDA 11.8 toolkit already present, skipping."
else
    apt-get -qq update
    apt-get -qq install -y wget gnupg
    wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
    dpkg -i cuda-keyring_1.1-1_all.deb
    apt-get -qq update
    apt-get -qq install -y cuda-toolkit-11-8
fi

export PATH="/usr/local/cuda-11.8/bin:$PATH"
nvcc --version
echo '✅ CUDA 11.8 toolkit ready!'

🔧 Installing CUDA 11.8 Toolkit (nvcc)
✅ CUDA 11.8 toolkit already present, skipping.
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0
✅ CUDA 11.8 toolkit ready!


## 3. Clone the TRELLIS repository
`--recurse-submodules` is required — TRELLIS depends on a couple of git submodules.

In [4]:
%cd /content
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd /content/TRELLIS

/content
fatal: destination path 'TRELLIS' already exists and is not an empty directory.
/content/TRELLIS


## 3.5 Patch `setup.sh` so it actually installs a CUDA-enabled PyTorch

**Why this cell exists:** the official `setup.sh` installs torch for `--new-env` with:
```
conda install pytorch==2.4.0 torchvision==0.19.0 pytorch-cuda=11.8 -c pytorch -c nvidia
```
Inside Colab's `condacolab` (Miniforge-based) environment, `conda-forge` is already a default
channel, and conda's solver can silently satisfy the `pytorch` package from a non-CUDA build even
though `pytorch-cuda=11.8` was requested. The result is a CPU-only torch that everything else
(`kaolin`, `nvdiffrast`, `diffoctreerast`, ...) then gets built against — which is exactly the
`AssertionError('Torch not compiled with CUDA enabled')` error.

The fix: rewrite that one line in the cloned `setup.sh` to install torch via `pip` pinned straight
at the official CUDA 11.8 wheel index. `pip` can't silently substitute a CPU build the way conda's
solver can. This must run **after** Step 3 (clone) and **before** Step 6 (running `setup.sh`).

In [ ]:
%%bash
set -e
cd /content/TRELLIS

OLD='conda install pytorch==2.4.0 torchvision==0.19.0 pytorch-cuda=11.8 -c pytorch -c nvidia'
NEW='pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu118'

if ! grep -qF "$OLD" setup.sh; then
    echo "⚠️  Expected line not found in setup.sh — it may have changed upstream."
    echo "    Open setup.sh and check the torch install line inside the --new-env block manually."
    exit 1
fi

sed -i "s|$OLD|$NEW|" setup.sh

echo '✅ Patched. New torch install line in setup.sh:'
grep -n 'torch==2.4.0' setup.sh

## 4. Install `condacolab`

⚠️ **This cell restarts the Colab kernel automatically as soon as it finishes — that's normal.**
When it restarts, don't re-run this cell. Just continue running the notebook from **Step 5** below.

In [5]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


---
## ⬇️ Kernel restarted? Continue from here. ⬇️
---
## 5. Verify conda is available and accept the Anaconda channel Terms of Service

Recent conda versions require explicitly accepting ToS for the default `main`/`r` channels before
installing anything from them non-interactively — this only needs to happen once per session.

In [6]:
import subprocess

def is_conda_installed():
    try:
        result = subprocess.run(['conda', '--version'], capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        return False

assert is_conda_installed(), "❌ Conda not found — did the kernel restart after condacolab.install()? Try Runtime > Restart session, then re-run from this cell."
print("✅ Conda is ready:", subprocess.check_output(['conda', '--version']).decode().strip())

✅ Conda is ready: conda 26.3.2


In [7]:
%%bash
set -e
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true
echo "✅ Channel ToS accepted"


✅ Channel ToS accepted


usage: conda [-h] [-v] [--no-plugins] [-V] COMMAND ...
conda: error: argument COMMAND: invalid choice: 'tos' (choose from activate, check, clean, commands, compare, config, create, deactivate, doctor, env, export, info, init, install, list, menuinst, notices, package, remove, rename, repoquery, run, search, uninstall, update, upgrade)
usage: conda [-h] [-v] [--no-plugins] [-V] COMMAND ...
conda: error: argument COMMAND: invalid choice: 'tos' (choose from activate, check, clean, commands, compare, config, create, deactivate, doctor, env, export, info, init, install, list, menuinst, notices, package, remove, rename, repoquery, run, search, uninstall, update, upgrade)


## 6. Run the official `setup.sh` with `--new-env`

This is the key fix versus a manual/base-env install:

- `--new-env` makes the script create its **own** `trellis` conda environment with **Python 3.10**
  and install the exact pinned `pytorch==2.4.0 torchvision==0.19.0 pytorch-cuda=11.8` combo TRELLIS
  is built and tested against — completely independent of Colab's Python 3.13 base env.
- Because that env is Python 3.10, packages like `open3d` (which has no 3.13 wheels) install cleanly.
- Flags used below match the official README's full-featured install, plus `--demo` for the Gradio app.

This step compiles several CUDA extensions from source and will take **~20–40 minutes**. This is
expected — TRELLIS's README calls this out explicitly ("installation may take a while").

In [9]:
%%bash
set -e

export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:${LD_LIBRARY_PATH:-}"
export CUDA_HOME="/usr/local/cuda-11.8"
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6+PTX"

# Required: hooks `conda activate` into this non-interactive shell
source /usr/local/etc/profile.d/conda.sh

cd /content/TRELLIS

echo '========================================'
echo '🏗️  Creating the "trellis" conda env (Python 3.10) and running setup.sh'
echo '    FLAGS: --new-env --basic --xformers --flash-attn'
echo '           --diffoctreerast --spconv --mipgaussian'
echo '           --kaolin --nvdiffrast --demo'
echo '========================================'

. ./setup.sh \
    --new-env \
    --basic \
    --xformers \
    --flash-attn \
    --diffoctreerast \
    --spconv \
    --mipgaussian \
    --kaolin \
    --nvdiffrast \
    --demo

echo ''
echo '✅ setup.sh complete! The "trellis" conda env is fully set up.'

🏗️  Creating the "trellis" conda env (Python 3.10) and running setup.sh
    FLAGS: --new-env --basic --xformers --flash-attn
           --diffoctreerast --spconv --mipgaussian
           --kaolin --nvdiffrast --demo
Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done

## Package Plan ##

  environment location: /usr/local/envs/trellis

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_10 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.7.22-hbd8a1cb_0 
  icu                conda-forge/linux-64::icu-78.3-py310h44b86e0_2 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.46.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.8.1-hecca717_1 
  libffi             conda-forge/linux-64::libffi-3.7.0-h81df57d_1 
  libgcc             conda-forge

  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /tmp/pip-req-build-c992znyq
  Running command git rev-parse -q --verify '9a4eb15e4021b67b12c460c7057d642626897ec8^{commit}'
  9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command git checkout -q 9a4eb15e4021b67b12c460c7057d642626897ec8


In [19]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

python -c "
import torch
try:
    torch.cuda.init()
    print('CUDA initialized OK')
    print(torch.cuda.get_device_name(0))
except Exception as e:
    print('CUDA init failed:', repr(e))
"

echo ''
echo '=== Also check the base env for comparison ==='
conda deactivate
python -c "
import torch
print('base env torch:', torch.__version__)
print('base env CUDA available:', torch.cuda.is_available())
" 2>&1 || echo "(base env has no torch — that's fine, just checking)"

CUDA init failed: AssertionError('Torch not compiled with CUDA enabled')

=== Also check the base env for comparison ===
Traceback (most recent call last):
  File "<string>", line 2, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'
(base env has no torch — that's fine, just checking)


## 7. Verify the install

Every command from here on must run **inside the `trellis` env**, not Colab's base env — use
`conda run -n trellis ...` (as below) for one-off commands.

In [12]:
%%bash
conda run -n trellis python -c "
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
import open3d
print('open3d:', open3d.__version__)
"

PyTorch: 2.4.0
CUDA available: False
open3d: 0.19.0


## 8. Run a minimal image-to-3D example

Uses the repo's own `assets/example_image/T.png`. Swap in your own image path as needed.

In [13]:
%%bash
set -e
cd /content/TRELLIS
export PATH="/usr/local/cuda-11.8/bin:$PATH"

conda run -n trellis python example.py

ls -la sample_gs.mp4 sample_rf.mp4 sample_mesh.mp4 sample.glb sample.ply 2>/dev/null || true
echo '✅ Example run complete — check /content/TRELLIS for the output files.'

[SPARSE] Backend: spconv, Attention: flash_attn


[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.4.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Traceback (most recent call last):
  File "/content/TRELLIS/example.py", line 9, in <module>
    from trellis.pipelines import TrellisImageTo3DPipeline
  File "/content/TRELLIS/trellis/__init__.py", line 5, in <module>
    from . import representations
  File "/content/TRELLIS/trellis/representations/__init__.py", line 4, in <module>
    from .mesh import MeshExtractResult
  File "/content/TRELLIS/trellis/representations/mesh/__init__.py", line 1, in <module>
    from .cube2mesh import SparseFeatures2Mesh, MeshExtractResult
  File "/content/TRELLIS/trellis/representations/mesh/cube2mesh.py", line 5, in <module>
    from .flexicubes.flexicubes import FlexiCubes
  File "/content/TRELLIS/trellis/representations/mesh/flexicubes/flexicubes.py", line 17, in <module>
    from 

CalledProcessError: Command 'b'set -e\ncd /content/TRELLIS\nexport PATH="/usr/local/cuda-11.8/bin:$PATH"\n\nconda run -n trellis python example.py\n\nls -la sample_gs.mp4 sample_rf.mp4 sample_mesh.mp4 sample.glb sample.ply 2>/dev/null || true\necho \'\xe2\x9c\x85 Example run complete \xe2\x80\x94 check /content/TRELLIS for the output files.\'\n'' returned non-zero exit status 1.

## 9. (Optional) Launch the Gradio web demo

Runs `app.py` with a public `share=True` link so you can use it from the browser. Stop the cell to
shut the demo down.

If your GPU doesn't support `flash-attn` (e.g. an older T4/V100), uncomment the `ATTN_BACKEND` line
to fall back to `xformers` — see the official README's note on this.

In [ ]:
%%bash
cd /content/TRELLIS
export PATH="/usr/local/cuda-11.8/bin:$PATH"

# export ATTN_BACKEND=xformers   # uncomment if your GPU doesn't support flash-attn

conda run -n trellis python app.py --server_port 7860 --share